##Setting up project directory

In [ ]:
%pip -q install pyBKT scikit-learn pandas numpy matplotlib seaborn fastapi "uvicorn[standard]" pydantic pytest

In [ ]:
import pandas as pd
import numpy as np
import sklearn
import fastapi
import pydantic
import pytest
import pyBKT

print("All imports successful")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

All imports successful
Pandas: 2.2.3
NumPy: 2.1.3
Scikit-learn: 1.6.1


In [ ]:
from pathlib import Path

folders = [
    "data/raw/ednet",
    "data/raw/assistments",
    "data/processed",
    "preprocessing",
    "models",
    "baselines",
    "scheduler",
    "evaluation",
    "experiments",
    "backend",
    "mobile",
    "eval_dashboard",
    "notebooks",
    "configs",
    "tests",
    "docs"
]

ROOT = Path("/content/REMEM")

for folder in folders:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

print("REMEM folder structure created")
print(ROOT)


REMEM folder structure created
/content/REMEM


In [ ]:
!find /content/REMEM -maxdepth 3 -type d | sort

/content/REMEM
/content/REMEM/backend
/content/REMEM/baselines
/content/REMEM/configs
/content/REMEM/data
/content/REMEM/data/processed
/content/REMEM/data/raw
/content/REMEM/data/raw/assistments
/content/REMEM/data/raw/ednet
/content/REMEM/docs
/content/REMEM/eval_dashboard
/content/REMEM/evaluation
/content/REMEM/experiments
/content/REMEM/mobile
/content/REMEM/models
/content/REMEM/notebooks
/content/REMEM/preprocessing
/content/REMEM/scheduler
/content/REMEM/tests


##downloading & extracting dataset

In [ ]:
!pip install -q gdown

import gdown
import os
import zipfile

url = "https://drive.google.com/uc?id=1AmGcOs5U31wIIqvthn9ARqJMrMTFTcaw"
output = "/content/EdNet-KT1.zip"

gdown.download(
    url=url,
    output=output,
    quiet=False,
    resume=True
)

print("File exists:", os.path.exists(output))
if os.path.exists(output):
    print("File size:", round(os.path.getsize(output) / (1024**3), 2), "GB")
    print("Valid ZIP:", zipfile.is_zipfile(output))


Downloading...
From (original): https://drive.google.com/uc?id=1AmGcOs5U31wIIqvthn9ARqJMrMTFTcaw
From (redirected): https://drive.google.com/uc?id=1AmGcOs5U31wIIqvthn9ARqJMrMTFTcaw&confirm=t&uuid=c7c2e362-ef35-4b1f-9149-b51909b9771d
To: /content/EdNet-KT1.zip
100%|██████████| 1.20G/1.20G [00:13<00:00, 87.4MB/s]

File exists: True
File size: 1.12 GB
Valid ZIP: True


In [ ]:
import zipfile
import os

zip_path = "/content/EdNet-KT1.zip"
extract_path = "/content/REMEM/data/raw/EdNet-KT1"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("Extraction completed.")
print("Dataset folder:", extract_path)
print("Sample files:", os.listdir(extract_path)[:10])


Extraction completed.
Dataset folder: /content/REMEM/data/raw/EdNet-KT1
Sample files: ['KT1']


In [ ]:
import os

for root, dirs, files in os.walk(extract_path):
    if files:
        print("Folder:", root)
        print("Sample files:", files[:5])
        break

Folder: /content/REMEM/data/raw/EdNet-KT1/KT1
Sample files: ['u614984.csv', 'u459981.csv', 'u163877.csv', 'u46468.csv', 'u483280.csv']


##sample user data

In [ ]:
import os
import pandas as pd

data_path = "/content/REMEM/data/raw/EdNet-KT1/KT1"

csv_files = [f for f in os.listdir(data_path) if f.endswith(".csv")]

print("Total user CSV files:", len(csv_files))

sample_file = os.path.join(data_path, csv_files[0])
df = pd.read_csv(sample_file)

print("\nSample file:", csv_files[0])
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)


Total user CSV files: 784309

Sample file: u614984.csv
Shape: (9, 5)

Columns:
['timestamp', 'solving_id', 'question_id', 'user_answer', 'elapsed_time']

First 5 rows:


,timestamp,solving_id,question_id,user_answer,elapsed_time
0,1560441501100,1,q3637,c,39000
1,1560441529911,2,q4655,b,21000
2,1560441558782,3,q326,b,25000
3,1560441579235,4,q3939,c,17000
4,1560441628306,5,q6273,c,43000



Data types:
timestamp        int64
solving_id       int64
question_id     object
user_answer     object
elapsed_time     int64
dtype: object


In [ ]:
required_columns = [
    "timestamp",
    "solving_id",
    "question_id",
    "user_answer",
    "elapsed_time"
]

print("Available columns:", df.columns.tolist())
print("\nColumn check:")

for col in required_columns:
    print(f"{col}: {'Present' if col in df.columns else 'Missing'}")


Available columns: ['timestamp', 'solving_id', 'question_id', 'user_answer', 'elapsed_time']

Column check:
timestamp: Present
solving_id: Present
question_id: Present
user_answer: Present
elapsed_time: Present


In [ ]:
##Step 6-7: Create and save the sample for REMEM

import os
import pandas as pd

data_path = "/content/REMEM/data/raw/EdNet-KT1/KT1"
output_path = "/content/REMEM/data/processed/ednet_sample.csv"

csv_files = sorted([
    f for f in os.listdir(data_path)
    if f.endswith(".csv")
])

sample_files = csv_files[:100]
data = []

for file in sample_files:
    file_path = os.path.join(data_path, file)
    df = pd.read_csv(file_path)
    df["student_id"] = file.replace(".csv", "")
    data.append(df)

sample_data = pd.concat(data, ignore_index=True)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
sample_data.to_csv(output_path, index=False)

print("Users sampled:", len(sample_files))
print("Total interactions:", len(sample_data))
print("Saved to:", output_path)
print("File size:", round(os.path.getsize(output_path) / (1024 * 1024), 2), "MB")

display(sample_data.head())

Users sampled: 100
Total interactions: 30819
Saved to: /content/REMEM/data/processed/ednet_sample.csv
File size: 1.15 MB


,timestamp,solving_id,question_id,user_answer,elapsed_time,student_id
0,1565096190868,1,q5012,b,38000,u1
1,1565096221062,2,q4706,c,24000,u1
2,1565096293432,3,q4366,b,68000,u1
3,1565096339668,4,q4829,a,42000,u1
4,1565096401774,5,q6528,b,59000,u1


In [ ]:
##Step 8: Check EdNet question metadata for REMEM

import os

base_path = "/content/REMEM/data/raw/EdNet-KT1"

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.lower() in ["questions.csv", "question.csv"]:
            print("Found:", os.path.join(root, file))

In [ ]:
##Step 9: Download and extract EdNet question metadata

import os
import zipfile
import gdown

content_zip = "/content/EdNet-content.zip"
content_path = "/content/REMEM/data/raw/EdNet-content"

url = "https://drive.google.com/uc?id=117aYJAWG3GU48suS66NPaB82HwFj6xWS"

gdown.download(url, content_zip, quiet=False, fuzzy=True)

if os.path.exists(content_zip) and zipfile.is_zipfile(content_zip):
    os.makedirs(content_path, exist_ok=True)

    with zipfile.ZipFile(content_zip, "r") as z:
        z.extractall(content_path)

    print("\nExtraction successful.")

    for root, dirs, files in os.walk(content_path):
        for file in files:
            print(os.path.join(root, file))
else:
    print("Download failed: the file is not a valid ZIP.")


Downloading...
From: https://drive.google.com/uc?id=117aYJAWG3GU48suS66NPaB82HwFj6xWS
To: /content/EdNet-content.zip
100%|██████████| 174k/174k [00:00<00:00, 64.2MB/s]



Extraction successful.
/content/REMEM/data/raw/EdNet-content/__MACOSX/contents/._questions.csv
/content/REMEM/data/raw/EdNet-content/__MACOSX/contents/._coupons.csv
/content/REMEM/data/raw/EdNet-content/__MACOSX/contents/._lectures.csv
/content/REMEM/data/raw/EdNet-content/__MACOSX/contents/._payments.csv
/content/REMEM/data/raw/EdNet-content/__MACOSX/contents/._.DS_Store
/content/REMEM/data/raw/EdNet-content/contents/questions.csv
/content/REMEM/data/raw/EdNet-content/contents/.DS_Store
/content/REMEM/data/raw/EdNet-content/contents/coupons.csv
/content/REMEM/data/raw/EdNet-content/contents/lectures.csv
/content/REMEM/data/raw/EdNet-content/contents/payments.csv


In [ ]:
##Step 10: Inspect EdNet question metadata for REMEM

import pandas as pd

questions_path = "/content/REMEM/data/raw/EdNet-content/contents/questions.csv"

questions = pd.read_csv(questions_path)

print("Number of questions:", len(questions))
print("\nColumns:", questions.columns.tolist())
print("\nFirst 5 rows:")
display(questions.head())

print("\nData types:")
print(questions.dtypes)

Number of questions: 13169

Columns: ['question_id', 'bundle_id', 'explanation_id', 'correct_answer', 'part', 'tags', 'deployed_at']

First 5 rows:


,question_id,bundle_id,explanation_id,correct_answer,part,tags,deployed_at
0,q1,b1,e1,b,1,1;2;179;181,1558093217098
1,q2,b2,e2,a,1,15;2;182,1558093219720
2,q3,b3,e3,b,1,14;2;179;183,1558093222784
3,q4,b4,e4,b,1,9;2;179;184,1558093225357
4,q5,b5,e5,c,1,8;2;179;181,1558093228439



Data types:
question_id       object
bundle_id         object
explanation_id    object
correct_answer    object
part               int64
tags              object
deployed_at        int64
dtype: object


In [ ]:
##Step 11: Merge question metadata and calculate correctness

import os
import pandas as pd

sample_path = "/content/REMEM/data/processed/ednet_sample.csv"
questions_path = "/content/REMEM/data/raw/EdNet-content/contents/questions.csv"
output_path = "/content/REMEM/data/processed/ednet_sample_with_correct.csv"

sample_data = pd.read_csv(sample_path)
questions = pd.read_csv(questions_path)

questions = questions[
    ["question_id", "correct_answer", "part", "tags"]
].drop_duplicates(subset=["question_id"])

merged_data = sample_data.merge(
    questions,
    on="question_id",
    how="left",
    validate="many_to_one"
)

user_answer = merged_data["user_answer"].astype("string").str.strip().str.lower()
correct_answer = merged_data["correct_answer"].astype("string").str.strip().str.lower()

matched = user_answer.notna() & correct_answer.notna()

merged_data["correct"] = pd.Series(
    pd.NA, index=merged_data.index, dtype="Int8"
)

merged_data.loc[matched, "correct"] = (
    user_answer[matched] == correct_answer[matched]
).astype("int8")

os.makedirs(os.path.dirname(output_path), exist_ok=True)
merged_data.to_csv(output_path, index=False)

print("Total interactions:", len(merged_data))
print("Matched questions:", int(matched.sum()))
print("Unmatched questions:", int((~matched).sum()))
print("\nCorrectness distribution:")
print(merged_data["correct"].value_counts(dropna=False).sort_index())
print("\nSaved to:", output_path)

display(merged_data.head())

Total interactions: 30819
Matched questions: 30811
Unmatched questions: 8

Correctness distribution:
correct
0       10330
1       20481
<NA>        8
Name: count, dtype: Int64

Saved to: /content/REMEM/data/processed/ednet_sample_with_correct.csv


,timestamp,solving_id,question_id,user_answer,elapsed_time,student_id,correct_answer,part,tags,correct
0,1565096190868,1,q5012,b,38000,u1,c,5,74,0
1,1565096221062,2,q4706,c,24000,u1,c,5,71,1
2,1565096293432,3,q4366,b,68000,u1,b,5,103,1
3,1565096339668,4,q4829,a,42000,u1,c,5,83,0
4,1565096401774,5,q6528,b,59000,u1,d,5,90,0


In [ ]:
##Step 12: Check data quality for REMEM

import pandas as pd

data_path = "/content/REMEM/data/processed/ednet_sample_with_correct.csv"
df = pd.read_csv(data_path)

print("Total rows:", len(df))
print("Unique students:", df["student_id"].nunique())
print("Unique questions:", df["question_id"].nunique())
print("Duplicate rows:", df.duplicated().sum())

print("\nMissing values:")
print(df[["student_id", "question_id", "user_answer", "correct", "tags"]].isna().sum())

print("\nInteractions by student:")
print(df.groupby("student_id").size().describe().round(2))

Total rows: 30819
Unique students: 100
Unique questions: 8356
Duplicate rows: 0

Missing values:
student_id     0
question_id    0
user_answer    8
correct        8
tags           0
dtype: int64

Interactions by student:
count     100.00
mean      308.19
std       670.53
min         1.00
25%         7.75
50%        30.00
75%       107.50
max      3436.00
dtype: float64


In [ ]:
##Step 13: Prepare the clean REMEM dataset

import os
import pandas as pd

input_path = "/content/REMEM/data/processed/ednet_sample_with_correct.csv"
output_path = "/content/REMEM/data/processed/ednet_clean.csv"

df = pd.read_csv(input_path)

df = df.dropna(subset=["student_id", "question_id", "user_answer", "correct"]).copy()

df["correct"] = df["correct"].astype("int8")
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
df["elapsed_time"] = pd.to_numeric(df["elapsed_time"], errors="coerce")

df = df.dropna(subset=["timestamp", "elapsed_time"])

df = df.sort_values(
    ["student_id", "timestamp"]
).reset_index(drop=True)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)

print("Clean dataset saved:", output_path)
print("Total interactions:", len(df))
print("Unique students:", df["student_id"].nunique())
print("Unique questions:", df["question_id"].nunique())
print("Missing values in correctness:", df["correct"].isna().sum())

display(df.head())

Clean dataset saved: /content/REMEM/data/processed/ednet_clean.csv
Total interactions: 30811
Unique students: 100
Unique questions: 8356
Missing values in correctness: 0


,timestamp,solving_id,question_id,user_answer,elapsed_time,student_id,correct_answer,part,tags,correct
0,1565096190868,1,q5012,b,38000,u1,c,5,74,0
1,1565096221062,2,q4706,c,24000,u1,c,5,71,1
2,1565096293432,3,q4366,b,68000,u1,b,5,103,1
3,1565096339668,4,q4829,a,42000,u1,c,5,83,0
4,1565096401774,5,q6528,b,59000,u1,d,5,90,0


In [ ]:
##Step 14: Inspect knowledge tags for REMEM

import pandas as pd

data_path = "/content/REMEM/data/processed/ednet_clean.csv"
df = pd.read_csv(data_path)

print("Sample tag values:")
print(df["tags"].head(15).to_list())

print("\nUnique tag strings:", df["tags"].nunique())

all_tags = set()

for value in df["tags"].dropna().astype(str):
    all_tags.update(tag.strip() for tag in value.split(";") if tag.strip())

print("Unique individual tags:", len(all_tags))
print("Sample individual tags:", sorted(all_tags)[:30])

print("\nQuestions with multiple tags:",
      df["tags"].dropna().astype(str).str.contains(";").sum())

Sample tag values:
['74', '71', '103', '83', '90', '82', '77', '30;34;42;181;183', '24;43;184;181', '27;28;45;184;182', '27;44;49;181;184', '24;25;182;183', '31;32;185;181', '30;34;35;184;182', '30;44;49;184;181']

Unique tag strings: 1581
Unique individual tags: 187
Sample individual tags: ['-1', '1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '12', '120', '121', '122', '123', '124']

Questions with multiple tags: 16165


In [ ]:
##Step 15: Create the question-to-knowledge-tag mapping

import os
import pandas as pd

input_path = "/content/REMEM/data/processed/ednet_clean.csv"
output_path = "/content/REMEM/data/processed/ednet_question_tags.csv"

df = pd.read_csv(input_path)

question_tags = (
    df[["question_id", "tags"]]
    .dropna(subset=["tags"])
    .drop_duplicates()
    .assign(tag_id=lambda x: x["tags"].astype(str).str.split(";"))
    .explode("tag_id")
)

question_tags["tag_id"] = question_tags["tag_id"].str.strip()

question_tags = (
    question_tags[["question_id", "tag_id"]]
    .drop_duplicates()
    .sort_values(["question_id", "tag_id"])
    .reset_index(drop=True)
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
question_tags.to_csv(output_path, index=False)

print("Question-tag mapping saved:", output_path)
print("Question-tag associations:", len(question_tags))
print("Unique questions:", question_tags["question_id"].nunique())
print("Unique tags:", question_tags["tag_id"].nunique())
print("Tag IDs including -1:", (question_tags["tag_id"] == "-1").sum())

display(question_tags.head(10))

Question-tag mapping saved: /content/REMEM/data/processed/ednet_question_tags.csv
Question-tag associations: 19073
Unique questions: 8356
Unique tags: 187
Tag IDs including -1: 4


,question_id,tag_id
0,q1,1
1,q1,179
2,q1,181
3,q1,2
4,q10,17
5,q10,182
6,q10,7
7,q100,181
8,q100,2
9,q100,22


In [ ]:
##Step 16: Validate question-tag mapping for REMEM

import pandas as pd

clean_path = "/content/REMEM/data/processed/ednet_clean.csv"
mapping_path = "/content/REMEM/data/processed/ednet_question_tags.csv"

df = pd.read_csv(clean_path)
question_tags = pd.read_csv(mapping_path, dtype={"tag_id": str})

known_questions = set(df["question_id"].unique())
mapped_questions = set(question_tags["question_id"].unique())

print("Questions in clean dataset:", len(known_questions))
print("Questions in mapping:", len(mapped_questions))
print("Mapped questions missing from clean dataset:",
      len(mapped_questions - known_questions))

print("\nRows with tag -1:")
display(question_tags[question_tags["tag_id"] == "-1"])

print("\nNumber of questions with at least one tag:",
      question_tags["question_id"].nunique())

Questions in clean dataset: 8356
Questions in mapping: 8356
Mapped questions missing from clean dataset: 0

Rows with tag -1:


,question_id,tag_id
4013,q16961,-1
4014,q16962,-1
4015,q16963,-1
13825,q6963,-1



Number of questions with at least one tag: 8356


In [ ]:
##Step 17: Create the final knowledge-component mapping

import os
import pandas as pd

input_path = "/content/REMEM/data/processed/ednet_question_tags.csv"
output_path = "/content/REMEM/data/processed/ednet_knowledge_components.csv"

question_tags = pd.read_csv(input_path, dtype={"tag_id": str})

knowledge_components = (
    question_tags[question_tags["tag_id"] != "-1"]
    .drop_duplicates()
    .rename(columns={"tag_id": "knowledge_component_id"})
    .sort_values(["question_id", "knowledge_component_id"])
    .reset_index(drop=True)
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
knowledge_components.to_csv(output_path, index=False)

print("Knowledge-component mapping saved:", output_path)
print("Question-component associations:", len(knowledge_components))
print("Unique questions:", knowledge_components["question_id"].nunique())
print("Unique components:", knowledge_components["knowledge_component_id"].nunique())
print("Remaining -1 tags:",
      (knowledge_components["knowledge_component_id"] == "-1").sum())

display(knowledge_components.head(10))

Knowledge-component mapping saved: /content/REMEM/data/processed/ednet_knowledge_components.csv
Question-component associations: 19069
Unique questions: 8352
Unique components: 186
Remaining -1 tags: 0


,question_id,knowledge_component_id
0,q1,1
1,q1,179
2,q1,181
3,q1,2
4,q10,17
5,q10,182
6,q10,7
7,q100,181
8,q100,2
9,q100,22


In [ ]:
##Step 18: Create a REMEM dataset summary

import os
import pandas as pd

data_path = "/content/REMEM/data/processed"
interactions = pd.read_csv(os.path.join(data_path, "ednet_clean.csv"))
components = pd.read_csv(os.path.join(data_path, "ednet_knowledge_components.csv"))

summary = {
    "total_interactions": len(interactions),
    "unique_students": interactions["student_id"].nunique(),
    "unique_questions": interactions["question_id"].nunique(),
    "correct_interactions": int((interactions["correct"] == 1).sum()),
    "incorrect_interactions": int((interactions["correct"] == 0).sum()),
    "unique_knowledge_components": components["knowledge_component_id"].nunique(),
    "question_component_associations": len(components)
}

summary_df = pd.DataFrame(
    list(summary.items()),
    columns=["metric", "value"]
)

summary_path = os.path.join(data_path, "ednet_summary.csv")
summary_df.to_csv(summary_path, index=False)

print("Dataset summary saved:", summary_path)
display(summary_df)

Dataset summary saved: /content/REMEM/data/processed/ednet_summary.csv


,metric,value
0,total_interactions,30811
1,unique_students,100
2,unique_questions,8356
3,correct_interactions,20481
4,incorrect_interactions,10330
5,unique_knowledge_components,186
6,question_component_associations,19069


In [ ]:
##Step 19: Download the ASSISTments dataset

!pip -q install gdown

import os
import gdown

assistments_path = "/content/REMEM/data/raw/assistments"
os.makedirs(assistments_path, exist_ok=True)

file_id = "1NNXHFRxcArrU0ZJSb9BIL56vmUt5FhlE"
output_path = os.path.join(
    assistments_path,
    "assistments_skill_builder_2009_2010.csv"
)

if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
    print("Dataset already downloaded:", output_path)
else:
    url = f"https://drive.google.com/uc?id={file_id}"
    result = gdown.download(url, output_path, quiet=False)

    if result and os.path.exists(output_path):
        print("Download completed:", output_path)
        print("File size:", round(os.path.getsize(output_path) / (1024 ** 2), 2), "MB")
    else:
        print("Download failed. Share the error message.")

Downloading...
From: https://drive.google.com/uc?id=1NNXHFRxcArrU0ZJSb9BIL56vmUt5FhlE
To: /content/REMEM/data/raw/assistments/assistments_skill_builder_2009_2010.csv
100%|██████████| 64.4M/64.4M [00:00<00:00, 205MB/s]

Download completed: /content/REMEM/data/raw/assistments/assistments_skill_builder_2009_2010.csv
File size: 61.43 MB


In [ ]:
##Step 20: Inspect the ASSISTments dataset

import pandas as pd

file_path = "/content/REMEM/data/raw/assistments/assistments_skill_builder_2009_2010.csv"

assistments = pd.read_csv(
    file_path,
    encoding="cp1252",
    low_memory=False
)

print("Dataset shape:", assistments.shape)

print("\nColumns:")
print(assistments.columns.tolist())

print("\nMissing values:")
print(assistments.isnull().sum())

print("\nFirst 5 rows:")
display(assistments.head())

Dataset shape: (346860, 31)

Columns:
['Unnamed: 0', 'order_id', 'assignment_id', 'user_id', 'assistment_id', 'problem_id', 'original', 'correct', 'attempt_count', 'ms_first_response', 'tutor_mode', 'answer_type', 'sequence_id', 'student_class_id', 'position', 'type', 'base_sequence_id', 'skill_id', 'skill_name', 'teacher_id', 'school_id', 'hint_count', 'hint_total', 'overlap_time', 'template_id', 'answer_id', 'answer_text', 'first_action', 'bottom_hint', 'opportunity', 'opportunity_original']

Missing values:
Unnamed: 0                   0
order_id                     0
assignment_id                0
user_id                      0
assistment_id                0
problem_id                   0
original                     0
correct                      0
attempt_count                0
ms_first_response            0
tutor_mode                   0
answer_type                  0
sequence_id                  0
student_class_id             0
position                     0
type               

,Unnamed: 0,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,...,hint_count,hint_total,overlap_time,template_id,answer_id,answer_text,first_action,bottom_hint,opportunity,opportunity_original
0,1,33022537,277618,64525,33139,51424,1,1,1,32454,...,0,3,32454,30799,NaN,26,0,NaN,1,1.0
1,2,33022709,277618,64525,33150,51435,1,1,1,4922,...,0,3,4922,30799,NaN,55,0,NaN,2,2.0
2,3,35450204,220674,70363,33159,51444,1,0,2,25390,...,0,3,42000,30799,NaN,88,0,NaN,1,1.0
3,4,35450295,220674,70363,33110,51395,1,1,1,4859,...,0,3,4859,30059,NaN,41,0,NaN,2,2.0
4,5,35450311,220674,70363,33196,51481,1,0,14,19813,...,3,4,124564,30060,NaN,65,0,0.0,3,3.0


In [ ]:
##Step 21: Validate the ASSISTments dataset

print("Total interactions:", len(assistments))
print("Unique students:", assistments["user_id"].nunique())
print("Unique problems:", assistments["problem_id"].nunique())
print("Unique assistments:", assistments["assistment_id"].nunique())
print("Unique skill IDs:", assistments["skill_id"].nunique())
print("Unique skill names:", assistments["skill_name"].nunique())

print("\nCorrectness values:")
print(assistments["correct"].value_counts(dropna=False))

print("\nMissing skill IDs:", assistments["skill_id"].isna().sum())
print("Missing skill names:", assistments["skill_name"].isna().sum())

print("\nSample skill mappings:")
display(
    assistments[["problem_id", "skill_id", "skill_name"]]
    .dropna(subset=["skill_id"])
    .drop_duplicates()
    .head(10)
)


Total interactions: 346860
Unique students: 4217
Unique problems: 26688
Unique assistments: 17725
Unique skill IDs: 149
Unique skill names: 101

Correctness values:
correct
1    223818
0    123042
Name: count, dtype: int64

Missing skill IDs: 63755
Missing skill names: 72270

Sample skill mappings:


,problem_id,skill_id,skill_name
0,51424,1_13,Box and Whisker
1,51435,1_13,Box and Whisker
2,51444,1_13,Box and Whisker
3,51395,1_13,Box and Whisker
4,51481,1_15,Box and Whisker
5,51457,1_15,Box and Whisker
6,51459,1_15,Box and Whisker
7,51408,1_13,Box and Whisker
8,51453,1_15,Box and Whisker
9,51397,1_13,Box and Whisker


In [ ]:

##Step 22: Clean and save the ASSISTments dataset

import os
import pandas as pd

output_path = "/content/REMEM/data/processed/assistments_clean.csv"

clean = assistments.copy()

clean = clean.rename(columns={
    "user_id": "student_id",
    "problem_id": "question_id",
    "skill_id": "knowledge_component_id",
    "skill_name": "knowledge_component_name"
})

clean = clean[
    [
        "student_id",
        "question_id",
        "correct",
        "order_id",
        "sequence_id",
        "knowledge_component_id",
        "knowledge_component_name",
        "attempt_count",
        "hint_count",
        "ms_first_response"
    ]
].copy()

clean["correct"] = pd.to_numeric(clean["correct"], errors="coerce")
clean = clean[clean["correct"].isin([0, 1])].copy()
clean["correct"] = clean["correct"].astype("int8")

clean = clean.sort_values(
    ["student_id", "order_id"]
).reset_index(drop=True)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
clean.to_csv(output_path, index=False)

print("Clean dataset saved:", output_path)
print("Total interactions:", len(clean))
print("Unique students:", clean["student_id"].nunique())
print("Unique questions:", clean["question_id"].nunique())
print("Unique knowledge components:", clean["knowledge_component_id"].nunique())
print("Rows without skill IDs:", clean["knowledge_component_id"].isna().sum())
print("Missing correctness:", clean["correct"].isna().sum())

display(clean.head())


Clean dataset saved: /content/REMEM/data/processed/assistments_clean.csv
Total interactions: 346860
Unique students: 4217
Unique questions: 26688
Unique knowledge components: 149
Rows without skill IDs: 63755
Missing correctness: 0


,student_id,question_id,correct,order_id,sequence_id,knowledge_component_id,knowledge_component_name,attempt_count,hint_count,ms_first_response
0,14,93383,0,21617623,7118,2_37_70,Circle Graph,1,2,26271
1,14,93407,1,21617632,7118,2_37_70,Circle Graph,1,0,29123
2,14,93400,0,21617641,7118,2_37_70,Circle Graph,1,2,13779
3,14,93419,0,21617650,7118,2_37_70,Circle Graph,1,2,16901
4,14,93420,0,21617659,7118,2_37_70,Circle Graph,1,2,11079


In [ ]:
##Step 23: Validate the saved ASSISTments dataset

import pandas as pd
import os

file_path = "/content/REMEM/data/processed/assistments_clean.csv"

check = pd.read_csv(file_path, low_memory=False)

print("File exists:", os.path.exists(file_path))
print("Dataset shape:", check.shape)
print("Missing student IDs:", check["student_id"].isna().sum())
print("Missing question IDs:", check["question_id"].isna().sum())
print("Missing correctness:", check["correct"].isna().sum())
print("Invalid correctness values:", (~check["correct"].isin([0, 1])).sum())
print("Duplicate rows:", check.duplicated().sum())

print("\nCorrectness distribution:")
print(check["correct"].value_counts().sort_index())

print("\nSaved columns:")
print(check.columns.tolist())


File exists: True
Dataset shape: (346860, 10)
Missing student IDs: 0
Missing question IDs: 0
Missing correctness: 0
Invalid correctness values: 0
Duplicate rows: 0

Correctness distribution:
correct
0    123042
1    223818
Name: count, dtype: int64

Saved columns:
['student_id', 'question_id', 'correct', 'order_id', 'sequence_id', 'knowledge_component_id', 'knowledge_component_name', 'attempt_count', 'hint_count', 'ms_first_response']
